# Wound Recovery Analysis — RVI & Pattern Classification

## 파이프라인
1. `cases/` 폴더에서 시계열 마스크 면적 추출
2. RVI (Recovery Velocity Index) 산출
3. 패턴 분류: Normal / Plateau / Rebound (룰 기반)
4. 회복 곡선 시각화

## 1. Config

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# ===== PATHS =====
CASES_DIR = "/kaggle/input/datasets/seoyeongb/wound-seg-data/data_wound_seg/cases"
OUT_DIR   = "/kaggle/working/rvi_out"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== CASE 구조 =====
DAYS = [0, 3, 6, 9, 12, 15, 18]          # 시계열 측정일
DAYS_FMT = [f"{d:02d}" for d in DAYS]    # 폴더명 형식: '00', '03', ...

# case_id → 패턴 타입 매핑
def get_case_type(case_id: int) -> str:
    if case_id <= 49:   return "normal"
    elif case_id <= 64: return "plateau"
    else:               return "rebound"

# ===== RVI 파라미터 =====
REF_DAYS       = 14    # 정규화 기준일 (2주)
REBOUND_THR    = 0.05  # 이 비율 이상 증가하면 rebound 의심
PLATEAU_THR    = 0.10  # 마지막 3구간 변화율 모두 이 이하이면 plateau

print(f"CASES_DIR 존재: {os.path.isdir(CASES_DIR)}")
print(f"케이스 폴더 수: {len(os.listdir(CASES_DIR))}")

## 2. 시계열 면적 추출

In [ ]:
def read_mask_area(mask_path: str) -> int:
    """
    마스크 파일에서 양성 픽셀 수(상처 면적)를 반환.

    Args:
        mask_path (str): 마스크 PNG 파일 경로
    Returns:
        int: 양성(>0) 픽셀 수. 파일 없으면 -1.
    """
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return -1
    return int((m > 127).sum())


def load_case_timeseries(cases_dir: str) -> pd.DataFrame:
    """
    cases/ 폴더를 순회하며 케이스별 시계열 면적 데이터를 DataFrame으로 반환.

    Returns:
        pd.DataFrame: 컬럼 = [case_id, case_type, day_00, day_03, ..., day_18]
    """
    records = []
    missing = 0

    for case_dir in sorted(Path(cases_dir).iterdir()):
        if not case_dir.is_dir():
            continue

        # case_0000 → case_id = 0
        name = case_dir.name  # e.g. 'case_0042'
        try:
            case_id = int(name.split("_")[1])
        except (IndexError, ValueError):
            continue

        case_type = get_case_type(case_id)
        row = {"case_id": case_id, "case_type": case_type}

        for day_str in DAYS_FMT:
            mask_path = str(case_dir / day_str / "mask.png")
            area = read_mask_area(mask_path)
            if area == -1:
                missing += 1
            row[f"day_{day_str}"] = area

        records.append(row)

    df = pd.DataFrame(records).sort_values("case_id").reset_index(drop=True)
    print(f"로드 완료: {len(df)}개 케이스, 누락 마스크: {missing}개")
    return df


ts_df = load_case_timeseries(CASES_DIR)
print(ts_df[["case_id", "case_type"]].groupby("case_type").count())
ts_df.head(3)

## 3. RVI 산출

$$\text{RVI} = \text{clip}\left(\frac{A_0 - A_{\text{last}}}{A_0} \times \frac{\text{REF\_DAYS}}{T_{\text{last}}} \times 100,\ 0,\ 100\right)$$

- $A_0$: 초기(Day 0) 면적, $A_{\text{last}}$: 최종 면적
- REF_DAYS = 14 (2주 기준으로 정규화)
- RVI 100 = 14일 안에 100% 회복, RVI 0 = 전혀 회복 안 됨

In [ ]:
def compute_rvi(areas: list, days: list, ref_days: int = REF_DAYS) -> float:
    """
    시계열 면적 데이터로부터 RVI(Recovery Velocity Index)를 산출.

    Args:
        areas (list[int]): 각 시점의 상처 면적 픽셀 수
        days (list[int]): 각 시점의 측정일
        ref_days (int): 정규화 기준일 (기본 14일)
    Returns:
        float: RVI 점수 (0~100)
    """
    a0, a_last = areas[0], areas[-1]
    t_elapsed  = days[-1] - days[0]

    if a0 <= 0 or t_elapsed <= 0:
        return 0.0

    total_reduction = (a0 - a_last) / a0   # 0~1 (음수 가능 = 악화)
    rvi = total_reduction / t_elapsed * ref_days * 100
    return float(np.clip(rvi, 0, 100))


def compute_rvi_from_row(row: pd.Series) -> float:
    """
    DataFrame 행(row)에서 RVI를 계산. area가 -1인 시점은 선형 보간.

    Args:
        row (pd.Series): day_00 ~ day_18 컬럼 포함 행
    Returns:
        float: RVI 점수
    """
    area_cols = [f"day_{d}" for d in DAYS_FMT]
    areas = [row[c] for c in area_cols]

    # -1(누락)은 이웃 값으로 선형 보간
    areas = pd.Series(areas, dtype=float).replace(-1, np.nan)\
              .interpolate(method="linear").fillna(method="bfill").fillna(method="ffill")\
              .tolist()

    return compute_rvi(areas, DAYS)


ts_df["rvi"] = ts_df.apply(compute_rvi_from_row, axis=1)

print("===== RVI 통계 =====")
print(ts_df.groupby("case_type")["rvi"].agg(["mean", "std", "min", "max"]).round(2))

## 4. 패턴 분류 (룰 기반)

| 패턴 | 규칙 |
|------|------|
| **Rebound** | 중간 구간(day 3~15)에서 5% 이상 면적 증가가 1회 이상 |
| **Plateau** | 후반 3개 구간(day 9→12, 12→15, 15→18) 변화율 모두 10% 미만 |
| **Normal** | 위 두 조건 모두 해당 없음 |

In [ ]:
def classify_pattern(areas: list,
                     rebound_thr: float = REBOUND_THR,
                     plateau_thr: float = PLATEAU_THR) -> str:
    """
    시계열 면적 데이터로부터 치유 패턴을 분류.

    Rules:
      Rebound : 중간 구간(인덱스 1~4)에서 면적이 rebound_thr 이상 증가
      Plateau : 마지막 3구간의 변화율이 모두 plateau_thr 미만
      Normal  : 위 두 조건 미해당

    Args:
        areas (list[float]): 시계열 면적 (길이 7)
        rebound_thr (float): rebound 판정 임계값 (기본 0.05)
        plateau_thr (float): plateau 판정 임계값 (기본 0.10)
    Returns:
        str: 'normal' | 'plateau' | 'rebound'
    """
    n = len(areas)

    # Rebound: 중간 구간(1~n-2)에서 증가 감지
    for i in range(1, n - 2):
        if areas[i] > 0 and (areas[i + 1] - areas[i]) / areas[i] > rebound_thr:
            return "rebound"

    # Plateau: 후반 3구간 변화율 모두 plateau_thr 미만
    late_changes = []
    for i in range(n - 4, n - 1):  # 인덱스 3,4,5 구간
        if areas[i] > 0:
            late_changes.append(abs(areas[i + 1] - areas[i]) / areas[i])
    if late_changes and all(c < plateau_thr for c in late_changes):
        return "plateau"

    return "normal"


def estimate_stable_day(areas: list, days: list, stable_thr: float = 0.05) -> int:
    """
    면적 변화율이 stable_thr 미만으로 처음 유지되는 날짜를 추정.

    Args:
        areas (list[float]): 시계열 면적
        days (list[int]): 측정일
        stable_thr (float): 안정 판정 임계값 (기본 5%)
    Returns:
        int: 안정화 추정일 (해당 없으면 마지막 측정일)
    """
    for i in range(len(areas) - 1):
        if areas[i] > 0 and abs(areas[i + 1] - areas[i]) / areas[i] < stable_thr:
            return days[i + 1]
    return days[-1]


def get_areas_from_row(row: pd.Series) -> list:
    area_cols = [f"day_{d}" for d in DAYS_FMT]
    areas = [float(row[c]) for c in area_cols]
    # 누락(-1)은 보간
    s = pd.Series(areas).replace(-1, np.nan)\
         .interpolate("linear").fillna(method="bfill").fillna(method="ffill")
    return s.tolist()


ts_df["pred_pattern"]  = ts_df.apply(lambda r: classify_pattern(get_areas_from_row(r)), axis=1)
ts_df["stable_day"]    = ts_df.apply(lambda r: estimate_stable_day(get_areas_from_row(r), DAYS), axis=1)

# ===== 분류 정확도 =====
correct = (ts_df["case_type"] == ts_df["pred_pattern"]).sum()
total   = len(ts_df)
print(f"분류 정확도: {correct}/{total} = {correct/total*100:.1f}%")
print()

# Confusion matrix
conf = pd.crosstab(ts_df["case_type"], ts_df["pred_pattern"],
                   rownames=["실제"], colnames=["예측"])
print(conf)

## 5. 회복 곡선 시각화

In [ ]:
# 패턴별 대표 케이스 3개씩 선택 (RVI 중앙값 근처)
PATTERNS      = ["normal", "plateau", "rebound"]
COLORS        = {"normal": "#2ecc71", "plateau": "#3498db", "rebound": "#e74c3c"}
SAMPLES_PER   = 5  # 패턴별 곡선 수

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

area_cols = [f"day_{d}" for d in DAYS_FMT]

for ax, pattern in zip(axes, PATTERNS):
    sub = ts_df[ts_df["case_type"] == pattern].copy()
    # RVI 중앙값 근처 케이스 선택
    median_rvi = sub["rvi"].median()
    sub["rvi_dist"] = (sub["rvi"] - median_rvi).abs()
    samples = sub.nsmallest(SAMPLES_PER, "rvi_dist")

    for _, row in samples.iterrows():
        areas  = [row[c] for c in area_cols]
        # 정규화: Day 0 기준 비율
        a0     = areas[0] if areas[0] > 0 else 1
        norm   = [a / a0 for a in areas]
        ax.plot(DAYS, norm, color=COLORS[pattern], alpha=0.6, linewidth=1.5)

    # 평균 곡선
    mean_norm = [
        (sub[c] / sub[area_cols[0]]).mean()
        for c in area_cols
    ]
    ax.plot(DAYS, mean_norm, color=COLORS[pattern], linewidth=3,
            linestyle="--", label=f"mean (n={len(sub)})")

    ax.set_title(f"{pattern.upper()}\n평균 RVI: {sub['rvi'].mean():.1f}",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Day")
    ax.set_ylabel("정규화 면적 (Day 0 = 1.0)")
    ax.set_ylim(0, 1.3)
    ax.axhline(1.0, color="gray", linestyle=":", alpha=0.5)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("패턴별 회복 곡선 (GT 마스크 면적 기반)", fontsize=15, y=1.02)
plt.tight_layout()
curve_path = os.path.join(OUT_DIR, "recovery_curves_by_pattern.png")
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print("저장 ->", curve_path)

## 6. RVI 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1) RVI 박스플롯 (패턴별)
ax = axes[0]
data_by_pattern = [ts_df[ts_df["case_type"] == p]["rvi"].values for p in PATTERNS]
bp = ax.boxplot(data_by_pattern, labels=[p.upper() for p in PATTERNS],
                patch_artist=True)
for patch, p in zip(bp["boxes"], PATTERNS):
    patch.set_facecolor(COLORS[p])
    patch.set_alpha(0.7)
ax.set_ylabel("RVI")
ax.set_title("패턴별 RVI 분포")
ax.grid(axis="y", alpha=0.3)

# 2) Confusion Matrix 히트맵
ax = axes[1]
conf_mat = pd.crosstab(ts_df["case_type"], ts_df["pred_pattern"])
conf_mat = conf_mat.reindex(index=PATTERNS, columns=PATTERNS, fill_value=0)

im = ax.imshow(conf_mat.values, cmap="Blues")
ax.set_xticks(range(len(PATTERNS)))
ax.set_yticks(range(len(PATTERNS)))
ax.set_xticklabels([p.upper() for p in PATTERNS])
ax.set_yticklabels([p.upper() for p in PATTERNS])
ax.set_xlabel("예측 패턴")
ax.set_ylabel("실제 패턴")
ax.set_title("Confusion Matrix")
for i in range(len(PATTERNS)):
    for j in range(len(PATTERNS)):
        val = conf_mat.values[i, j]
        ax.text(j, i, str(val), ha="center", va="center",
                fontsize=14, fontweight="bold",
                color="white" if val > conf_mat.values.max() * 0.5 else "black")
plt.colorbar(im, ax=ax)

plt.suptitle("RVI 분포 & 패턴 분류 결과", fontsize=14)
plt.tight_layout()
dist_path = os.path.join(OUT_DIR, "rvi_distribution_confusion.png")
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print("저장 ->", dist_path)

## 7. 개별 케이스 리포트 (예시)

In [ ]:
def print_case_report(row: pd.Series):
    """
    단일 케이스의 회복 분석 리포트를 출력.

    Args:
        row (pd.Series): ts_df의 단일 행
    """
    areas  = get_areas_from_row(row)
    a0     = areas[0]
    a_last = areas[-1]
    total_reduction = (a0 - a_last) / a0 * 100 if a0 > 0 else 0

    print(f"케이스 ID     : case_{row['case_id']:04d}")
    print(f"실제 패턴     : {row['case_type']}")
    print(f"예측 패턴     : {row['pred_pattern']}")
    print(f"RVI           : {row['rvi']:.1f} / 100")
    print(f"초기 면적     : {int(a0):,} px")
    print(f"최종 면적     : {int(a_last):,} px")
    print(f"총 감소율     : {total_reduction:.1f}%")
    print(f"안정화 추정일 : Day {row['stable_day']}")
    print(f"시계열 면적   : {[int(a) for a in areas]}")
    print()


# 패턴별 대표 케이스 1개씩 출력
for pattern in PATTERNS:
    sample_row = ts_df[ts_df["case_type"] == pattern].iloc[0]
    print_case_report(sample_row)

# 전체 결과 저장
result_cols = ["case_id", "case_type", "pred_pattern", "rvi", "stable_day"] + \
              [f"day_{d}" for d in DAYS_FMT]
ts_df[result_cols].to_csv(os.path.join(OUT_DIR, "rvi_results.csv"),
                           index=False, encoding="utf-8-sig")
print("전체 결과 저장 ->", os.path.join(OUT_DIR, "rvi_results.csv"))

## 8. 전체 요약

In [ ]:
correct = (ts_df["case_type"] == ts_df["pred_pattern"]).sum()
total   = len(ts_df)

print("==========================================")
print("       Wound Recovery Analysis 요약")
print("==========================================")
print(f"분석 케이스 수     : {total}")
print(f"패턴 분류 정확도   : {correct}/{total} ({correct/total*100:.1f}%)")
print()
print("[RVI 통계 (패턴별)]")
print(ts_df.groupby("case_type")[["rvi"]].agg(["mean","std","min","max"]).round(1).to_string())
print()
print("[안정화 추정일 통계 (패턴별)]")
print(ts_df.groupby("case_type")[["stable_day"]].agg(["mean","min","max"]).round(1).to_string())
print()
print("[Confusion Matrix]")
print(pd.crosstab(ts_df["case_type"], ts_df["pred_pattern"],
                  rownames=["실제"], colnames=["예측"]))
print("==========================================")